# DDoS 3단계 — 임계값 룰 vs AI 탐지 비교

**목표**: 서버 관리자가 실제로 거는 **"IP당 초당 연결 수 · 동시 연결 수" 임계값 룰**과 AI 모델이
같은 판정 단위(**출발지 IP × 10초 구간**)에서 DoS/DDoS를 얼마나 잡고, 정상 사용자를 얼마나 막는지 비교한다.

| 구분 | 내용 |
|---|---|
| 데이터 | CIC-IDS2017 원본 pcap 이틀치 — 수요일(DoS Hulk·GoldenEye·slowloris·Slowhttptest), 금요일(DDoS LOIT) |
| 룰 기반 | iptables `hashlimit`(초당 새 연결 수) · `connlimit`(동시 연결 수)와 같은 방식의 임계값 2개 |
| AI 모델 A | 10초 구간 통계 특성 + 그래디언트 부스팅 (CPU) |
| AI 모델 B | 같은 특성 + MLP (GPU) |

**왜 이 단위인가**: 임계값 룰은 "이 IP가 지금 너무 많이 붙는다 → 이 IP를 막는다"로 동작합니다.
그래서 Suricata로 pcap을 흐름(연결)으로 바꾼 뒤, **출발지 IP별로 10초씩 묶어** 한 구간을 하나의 판정 대상으로 삼습니다.

**실행 전**: 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 모두 실행. 약 20–30분 (pcap 2개 22 GB 내려받기·분석). 디스크 약 25 GB 사용.

## 1. 환경 확인과 설정

In [ ]:
import os, sys, re, json, time, random, shutil, subprocess, platform, hashlib, glob, zipfile
IN_COLAB = 'google.colab' in sys.modules
DAYS = [d.strip() for d in os.environ.get('DDOS_DAYS', 'Wednesday,Friday').split(',') if d.strip()]
WIN = int(os.environ.get('DDOS_WIN', 10))          # 판정 구간(초)
WORK = '/content/ddos-lab' if IN_COLAB else os.path.abspath(os.environ.get('DDOS_LAB_DIR', 'ddos-lab'))
DOS_LABELS = {'DoS Hulk', 'DoS GoldenEye', 'DoS slowloris', 'DoS Slowhttptest', 'DDoS'}
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
RESULTS = os.path.join(WORK, 'results')
os.makedirs(RESULTS, exist_ok=True)

def sh(cmd, check=True, show=True, tail=3000):
    """셸 명령 실행 (Colab·로컬 공통)"""
    if show:
        print('$', cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    out = (r.stdout + r.stderr).strip()
    if show and out:
        print(out[-tail:])
    if check and r.returncode != 0:
        raise RuntimeError(f'명령 실패 (코드 {r.returncode}): {cmd}')
    return r

if IN_COLAB:
    sh('pip -q install orjson "huggingface_hub>=0.32" hf_xet', check=False, show=False)
import numpy as np, pandas as pd, sklearn, torch, orjson
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
VERSIONS = {
    'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
    'scikit-learn': sklearn.__version__, 'torch': torch.__version__,
    'gpu': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'none (CPU)',
}
print('분석 요일:', DAYS, f'/ 판정 구간 {WIN}초')
print(json.dumps(VERSIONS, indent=2, ensure_ascii=False))
_ = sh('nproc; free -g | head -2; df -h / | tail -1', check=False)

## 2. Suricata 설치
이번에는 탐지 룰을 쓰지 않고, Suricata를 **pcap → 연결(흐름) 기록 변환기**로만 씁니다.
시작·끝 시각, 방향별 패킷·바이트, TCP 플래그·상태가 흐름마다 기록됩니다.

In [ ]:
if not shutil.which('suricata'):
    sh('apt-get -qq update', check=False, show=False)
    sh('add-apt-repository -y ppa:oisf/suricata-stable', check=False, tail=300)
    sh('DEBIAN_FRONTEND=noninteractive apt-get -qq install -y suricata', tail=300)
VERSIONS['suricata'] = sh('suricata -V').stdout.strip().split('version')[-1].strip()

import yaml
cfg = yaml.safe_load(open('/etc/suricata/suricata.yaml'))
cfg['vars']['address-groups']['HOME_NET'] = '[192.168.10.0/24]'
for o in cfg['outputs']:
    k = list(o)[0]
    if k == 'eve-log':
        o[k].update({'enabled': True, 'filetype': 'regular', 'filename': 'eve.json', 'types': ['flow']})
    elif isinstance(o[k], dict):
        o[k]['enabled'] = False
cfg.setdefault('flow', {})['memcap'] = '2gb'
cfg.setdefault('stream', {})['memcap'] = '1gb'
cfg['stream']['midstream'] = True
cfg['stream'].setdefault('reassembly', {})['memcap'] = '2gb'
CFG = os.path.join(WORK, 'suricata-flows.yaml')
with open(CFG, 'w') as f:
    f.write('%YAML 1.1\n---\n' + yaml.safe_dump(cfg, sort_keys=False))
EMPTY_RULES = os.path.join(WORK, 'no.rules'); open(EMPTY_RULES, 'w').close()

## 3. 데이터 받기 (CIC-IDS2017)
2단계와 같은 Hugging Face 공개 미러(`bencorn/CICIDS2017`)에서 요일별 pcap과 라벨 파일을 받습니다.
공식 배포처: https://www.unb.ca/cic/datasets/ids-2017.html

In [ ]:
DATA = os.path.join(WORK, 'data'); os.makedirs(DATA, exist_ok=True)
LOCAL = json.loads(os.environ.get('DDOS_LOCAL', '{}'))   # 로컬 시험용: {"요일": {"pcap": 경로, "labels": 폴더}}
SRC = {}
if all(d in LOCAL for d in DAYS):
    SRC = {d: (LOCAL[d]['pcap'], LOCAL[d]['labels']) for d in DAYS}
else:
    from huggingface_hub import list_repo_files, hf_hub_download
    REPO = 'bencorn/CICIDS2017'
    files = list_repo_files(REPO, repo_type='dataset')
    zip_name = [f for f in files if os.path.basename(f).lower() == 'generatedlabelledflows.zip'][0]
    zpath = hf_hub_download(REPO, zip_name, repo_type='dataset', local_dir=DATA)
    for d in DAYS:
        t0 = time.time()
        pcap_name = [f for f in files if os.path.basename(f).lower() == f'{d.lower()}-workinghours.pcap'][0]
        pcap = hf_hub_download(REPO, pcap_name, repo_type='dataset', local_dir=DATA)
        ldir = os.path.join(DATA, f'labels-{d}')
        with zipfile.ZipFile(zpath) as z:
            for n in z.namelist():
                if n.lower().endswith('.csv') and os.path.basename(n).lower().startswith(d.lower()):
                    z.extract(n, ldir)
        SRC[d] = (pcap, ldir)
        print(f'{d}: 내려받기 {time.time()-t0:.0f}초')
PCAP_GB = {}
for d, (pcap, ldir) in SRC.items():
    PCAP_GB[d] = round(os.path.getsize(pcap) / 1e9, 2)
    print(f'{d}: pcap {PCAP_GB[d]} GB, 라벨 {[os.path.basename(p) for p in glob.glob(os.path.join(ldir, "**", "*.csv"), recursive=True)]}')

## 4. pcap → 흐름 기록 (Suricata)

In [ ]:
SURI_SEC = {}
for d, (pcap, _) in SRC.items():
    out = os.path.join(WORK, f'flows-{d}')
    eve = os.path.join(out, 'eve.json')
    if not os.path.exists(eve) or os.path.getsize(eve) == 0:
        shutil.rmtree(out, ignore_errors=True); os.makedirs(out)
        t0 = time.time()
        sh(f'suricata -c {CFG} -S {EMPTY_RULES} -r {pcap} -l {out} -k none --runmode autofp', tail=600)
        json.dump({'sec': round(time.time() - t0, 1)}, open(os.path.join(out, 'timing.json'), 'w'))
    SURI_SEC[d] = json.load(open(os.path.join(out, 'timing.json')))['sec']
    print(f'{d}: Suricata {SURI_SEC[d]:.0f}초, eve.json {os.path.getsize(eve)/1e6:.0f} MB')

def flags(h):
    try:
        return int(h, 16)
    except (TypeError, ValueError):
        return 0

rows = []
for d in SRC:
    with open(os.path.join(WORK, f'flows-{d}', 'eve.json'), 'rb') as f:
        for line in f:
            e = orjson.loads(line)
            if e.get('event_type') != 'flow':
                continue
            fl, tcp = e.get('flow', {}), e.get('tcp', {})
            rows.append((d, e.get('src_ip'), e.get('src_port', 0), e.get('dest_ip'), e.get('dest_port', 0), e.get('proto'),
                         e.get('app_proto', 'none'), fl.get('pkts_toserver', 0), fl.get('pkts_toclient', 0),
                         fl.get('bytes_toserver', 0), fl.get('bytes_toclient', 0), fl.get('start'), fl.get('end'),
                         flags(tcp.get('tcp_flags_ts')), flags(tcp.get('tcp_flags_tc'))))
FL = pd.DataFrame(rows, columns=['day', 'src_ip', 'src_port', 'dest_ip', 'dest_port', 'proto', 'app_proto', 'pkts_ts', 'pkts_tc',
                                 'bytes_ts', 'bytes_tc', 'start', 'end', 'flags_ts', 'flags_tc'])
del rows
EPOCH = pd.Timestamp('1970-01-01', tz='UTC')
FL['t0'] = (pd.to_datetime(FL.start, utc=True, format='ISO8601') - EPOCH).dt.total_seconds()
FL['t1'] = (pd.to_datetime(FL.end, utc=True, format='ISO8601') - EPOCH).dt.total_seconds()
print(f'흐름 {len(FL):,}개'); print(FL.groupby('day').size().to_string())

## 5. 흐름에 정답 라벨 붙이기
2단계와 같은 방법(양 끝 IP·포트 + 프로토콜)으로 CIC 라벨을 붙입니다. 요일이 다르면 같은 포트가 다시 쓰이므로 요일도 키에 넣습니다.
2단계에서 정상+DDoS가 섞인 "모호" 흐름이 많았는데, 공격자 IP의 연결이므로 이번에는 **공격 라벨을 우선**합니다.

In [ ]:
PROTO_NUM = {'TCP': 6, 'UDP': 17, 'ICMP': 1}
t_cell = time.time()
def norm_label(s):
    s = str(s).strip().replace('\ufffd', '-').replace('\x96', '-').replace('–', '-')
    return re.sub(r'\s+', ' ', s)
def key_frame(day, a_ip, a_port, b_ip, b_port, proto):
    """방향을 무시한 연결 키: 요일|작은쪽 IP:포트|큰쪽 IP:포트|프로토콜"""
    a = a_ip.astype(str).values + ':' + a_port.astype('int64').astype(str).values
    b = b_ip.astype(str).values + ':' + b_port.astype('int64').astype(str).values
    lo, hi = np.where(a <= b, a, b), np.where(a <= b, b, a)
    return pd.Series(day.astype(str).values + '|' + lo + '|' + hi + '|' + proto.astype('int64').astype(str).values, index=day.index)

parts = []
for d, (_, ldir) in SRC.items():
    for p in glob.glob(os.path.join(ldir, '**', '*.csv'), recursive=True):
        x = pd.read_csv(p, encoding='latin-1', low_memory=False, skipinitialspace=True,
                        usecols=lambda c: c.strip() in ('Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Label'))
        x.columns = [c.strip() for c in x.columns]
        x['day'] = d
        parts.append(x.dropna())
CIC = pd.concat(parts, ignore_index=True)
CIC['Label'] = CIC.Label.map(norm_label)
for c in ('Source Port', 'Destination Port', 'Protocol'):
    CIC[c] = pd.to_numeric(CIC[c], errors='coerce')
CIC = CIC.dropna()
print(f'CIC 라벨 {len(CIC):,}행 읽기 {time.time()-t_cell:.0f}초'); print(CIC.groupby(['day', 'Label']).size().to_string())
CIC['key'] = key_frame(CIC.day, CIC['Source IP'], CIC['Source Port'], CIC['Destination IP'], CIC['Destination Port'], CIC['Protocol'])
# 키별 라벨: 공격 라벨이 하나라도 있으면 그 공격(여러 개면 이름순 첫 번째), 아니면 BENIGN — 전부 벡터 연산
att = CIC.loc[CIC.Label != 'BENIGN', ['key', 'Label']].sort_values(['key', 'Label']).drop_duplicates('key')
KEY_LABEL = pd.Series('BENIGN', index=pd.Index(CIC.key.unique()))
KEY_LABEL.loc[att.key.values] = att.Label.values
del CIC, parts, att
FL['key'] = key_frame(FL.day, FL.src_ip, FL.src_port.fillna(0), FL.dest_ip, FL.dest_port.fillna(0), FL.proto.map(PROTO_NUM).fillna(0))
FL['label'] = FL.key.map(KEY_LABEL).fillna('UNMATCHED')
print(f'라벨 붙이기 끝 {time.time()-t_cell:.0f}초')
print('\n흐름 라벨:'); print(FL.groupby(['day', 'label']).size().to_string())

## 6. 출발지 IP × 10초 구간으로 묶기
한 구간 = "어떤 IP가 10초 동안 시작한 연결들". 구간마다 아래 값을 계산합니다.

| 값 | 뜻 | 쓰는 곳 |
|---|---|---|
| `srv_new_conn` | 10초 동안 **한 서버로** 새로 연 연결 수 (서버별 최댓값) | 룰 R1 (hashlimit 방식) |
| `srv_concurrent` | 구간이 끝나는 순간 **한 서버에** 열려 있는 연결 수 | 룰 R2 (connlimit 방식) |
| `new_conn`, `concurrent` | 목적지 구분 없이 센 값 | AI 특성 |
| 나머지 | 목적지 수, 포트 수, 응답 없는 연결 비율, 패킷·바이트·지속 시간 통계, HTTP 비율 등 | AI 특성 |

**구간 라벨은 CIC가 공개한 공격 일정표로 붙입니다.** 흐름 라벨을 IP·포트로 맞추면, 공격자가 하루 동안 같은 출발 포트를
여러 공격에 다시 써서 slowloris 흐름이 Hulk·GoldenEye로 잘못 붙는 문제가 있었습니다 (5번 출력 참고).
일정표(현지 시각, 캐나다 대서양 서머타임 UTC−3)와 공격자 172.16.0.1(NAT 뒤 주소) → 피해 서버 192.168.10.50 조건으로 붙입니다.

| 라벨 | 조건 |
|---|---|
| 그 공격 | 출발지가 공격자이고, 피해 서버로 간 연결이 있고, 구간이 공격 시간 안 |
| 제외(경계) | 공격자 구간 중 공격 시작·끝 ±2분 (시계 오차·잔여 연결) |
| 제외(다른 공격) | 포트 스캔·봇넷·Heartbleed 흐름이 있거나, 공격 시간 밖인데 DoS 흐름 라벨이 붙은 구간 |
| 정상 | 그 밖의 모든 구간 |

In [ ]:
SCHEDULE = json.loads(os.environ.get('DDOS_SCHEDULE', 'null')) or {
    'Wednesday': {'date': '2017-07-05', 'attacks': [['DoS slowloris', '09:47', '10:10'], ['DoS Slowhttptest', '10:14', '10:35'],
                                                    ['DoS Hulk', '10:43', '11:00'], ['DoS GoldenEye', '11:10', '11:23']]},
    'Friday': {'date': '2017-07-07', 'attacks': [['DDoS', '15:56', '16:16']]},
}
UTC_OFFSET_H = -3           # ADT
ATTACKER, VICTIM, EDGE = '172.16.0.1', '192.168.10.50', 120
def to_utc(day, hm):
    return (pd.Timestamp(f"{SCHEDULE[day]['date']} {hm}", tz='UTC') - EPOCH).total_seconds() - UTC_OFFSET_H * 3600
INTERVALS = [(d, atk, to_utc(d, a), to_utc(d, b) + 60) for d in SCHEDULE if d in SRC for atk, a, b in SCHEDULE[d]['attacks']]

F = FL.copy()
F['w'] = (F.t0 // WIN).astype('int64')
F['dur'] = (F.t1 - F.t0).clip(lower=0)
F['no_reply'] = (F.pkts_tc == 0).astype(int)
F['syn_only'] = ((F.flags_ts & 0x02) > 0) & ((F.flags_ts & 0x10) == 0)
F['rst'] = ((F.flags_tc | F.flags_ts) & 0x04) > 0
F['http'] = (F.app_proto == 'http').astype(int)
F['is_dos'] = F.label.isin(DOS_LABELS)
F['is_other'] = ~F.label.isin(DOS_LABELS | {'BENIGN', 'UNMATCHED'})
F['to_victim'] = F.dest_ip == VICTIM

# 일정표 검증: 공격 시간 안과 바로 앞 같은 길이 구간에서 공격자→피해 서버 연결 수
chk = []
a2v = F[(F.src_ip == ATTACKER) & F.to_victim]
for d, atk, t_s, t_e in INTERVALS:
    x = a2v[a2v.day == d]
    chk.append({'day': d, 'attack': atk, 'minutes': round((t_e - t_s) / 60), 'flows_in_window': int(((x.t0 >= t_s) & (x.t0 < t_e)).sum()),
                'flows_before': int(((x.t0 >= 2 * t_s - t_e) & (x.t0 < t_s)).sum())})
SCHED_CHECK = pd.DataFrame(chk)
print('일정표 검증 (공격자 → 피해 서버 연결 수):'); print(SCHED_CHECK.to_string(index=False))

g = F.groupby(['day', 'src_ip', 'w'])
W = g.agg(new_conn=('t0', 'size'), dst_ips=('dest_ip', 'nunique'), dst_ports=('dest_port', 'nunique'),
          no_reply=('no_reply', 'mean'), syn_only=('syn_only', 'mean'), rst=('rst', 'mean'), http=('http', 'mean'),
          pkts_ts_mean=('pkts_ts', 'mean'), pkts_tc_mean=('pkts_tc', 'mean'), bytes_ts_mean=('bytes_ts', 'mean'),
          bytes_tc_mean=('bytes_tc', 'mean'), bytes_ts_sum=('bytes_ts', 'sum'), dur_mean=('dur', 'mean'), dur_max=('dur', 'max'),
          dos_n=('is_dos', 'sum'), other_n=('is_other', 'sum'), to_victim=('to_victim', 'max')).reset_index()

# 동시 연결 수: 구간 끝 시각에 열려 있는 연결 (같은 IP 기준)
conc = np.zeros(len(W), dtype=np.int64)
grp_flows = {k: v for k, v in F.groupby(['day', 'src_ip'])[['t0', 't1']]}
for (d, ip), idx in W.groupby(['day', 'src_ip']).groups.items():
    fv = grp_flows[(d, ip)]
    s_, e_ = np.sort(fv.t0.values), np.sort(fv.t1.values)
    tend = (W.loc[idx, 'w'].values + 1) * WIN
    conc[np.asarray(idx)] = np.searchsorted(s_, tend, side='left') - np.searchsorted(e_, tend, side='left')
W['concurrent'] = conc

# 같은 서버(목적지 IP) 기준 값 — 서버에 거는 hashlimit/connlimit은 "자기에게 온 연결"만 셉니다.
F['pair'] = pd.factorize(F.day.astype(str) + '|' + F.src_ip.astype(str) + '|' + F.dest_ip.astype(str))[0]
pw = F.groupby(['pair', 'w']).agg(day=('day', 'first'), src_ip=('src_ip', 'first'), n=('t0', 'size')).reset_index()
# 구간 끝 시각에 열려 있는 연결 수(쌍별): 시작·끝 사건과 조회 시각을 한 번에 정렬해 누적합으로 계산
q_t = ((pw.w + 1) * WIN).values.astype('float64')
pair_all = np.concatenate([pw.pair.values, F.pair.values, F.pair.values])
time_all = np.concatenate([q_t, F.t0.values, F.t1.values])
kind_all = np.concatenate([np.zeros(len(pw)), np.ones(len(F)), np.full(len(F), 2)])   # 같은 시각이면 조회 → 시작 → 끝
delta = np.concatenate([np.zeros(len(pw)), np.ones(len(F)), -np.ones(len(F))])
order = np.lexsort((kind_all, time_all, pair_all))
d_sorted = delta[order]
cs = np.cumsum(d_sorted)
p_sorted = pair_all[order]
is_first = np.r_[True, p_sorted[1:] != p_sorted[:-1]]
start_idx = np.maximum.accumulate(np.where(is_first, np.arange(len(order)), 0))
cs_pair = cs - (cs[start_idx] - d_sorted[start_idx])          # 쌍마다 0부터 다시 센 누적합
qpos = np.flatnonzero(kind_all[order] == 0)
pw_conc = np.zeros(len(pw)); pw_conc[order[qpos]] = cs_pair[qpos]
pw['conc'] = pw_conc.astype(int)
srv = pw.groupby(['day', 'src_ip', 'w']).agg(srv_new_conn=('n', 'max'), srv_concurrent=('conc', 'max')).reset_index()
W = W.merge(srv, on=['day', 'src_ip', 'w'], how='left')

ws, we = W.w * WIN, (W.w + 1) * WIN
W['attack'] = 'BENIGN'
is_att = (W.src_ip == ATTACKER).values
in_any = np.zeros(len(W), bool)
for d, atk, t_s, t_e in INTERVALS:
    day = (W.day == d).values
    inside = day & is_att & W.to_victim.values & (ws >= t_s).values & (we <= t_e).values
    W.loc[inside, 'attack'] = atk
    near = day & is_att & (we > t_s - EDGE).values & (ws < t_e + EDGE).values
    W.loc[near & ~inside, 'attack'] = 'EXCLUDED_EDGE'
    in_any |= near
W.loc[(W.attack == 'BENIGN') & ((W.other_n > 0) | (W.dos_n > 0)), 'attack'] = 'EXCLUDED_OTHER'
print('\n구간 수:'); print(W.groupby(['day', 'attack']).size().to_string())
WINDOW_STATS = W.attack.value_counts().to_dict()
W = W[~W.attack.str.startswith('EXCLUDED')].reset_index(drop=True)
small = [a for a, n in W.attack.value_counts().items() if n < 10 and a != 'BENIGN']
if small:
    print('구간이 10개 미만이라 제외:', small); W = W[~W.attack.isin(small)].reset_index(drop=True)
W['label'] = (W.attack != 'BENIGN').astype(int)
print(f'\n비교에 쓰는 구간 {len(W):,}개 (정상 {int((W.label==0).sum()):,} / DoS·DDoS {int(W.label.sum()):,})')
print(W.groupby('attack')[['srv_new_conn', 'srv_concurrent', 'new_conn', 'concurrent', 'http', 'dur_mean']].median().round(2).to_string())

## 7. 학습 / 검증 / 시험 분할
공격 종류 비율을 유지한 채 60 / 20 / 20. 룰의 "튜닝한 임계값"도 AI 기준값과 똑같이 **검증 집합**에서 정합니다.

In [ ]:
from sklearn.model_selection import train_test_split
idx_tr, idx_tmp = train_test_split(W.index, test_size=0.4, stratify=W.attack, random_state=SEED)
idx_va, idx_te = train_test_split(idx_tmp, test_size=0.5, stratify=W.attack[idx_tmp], random_state=SEED)
W['split'] = 'train'; W.loc[idx_va, 'split'] = 'val'; W.loc[idx_te, 'split'] = 'test'
tr, va, te = (W.split == 'train').values, (W.split == 'val').values, (W.split == 'test').values
print(W.groupby(['split', 'attack']).size().unstack(fill_value=0).to_string())

## 8. 임계값 룰
**기본값**: 관리자가 흔히 거는 값 — 새 연결 **초당 10개**(10초에 100개) 이상, 또는 동시 연결 **50개** 이상이면 그 IP를 차단.
**튜닝값**: 검증 집합에서 정상 구간 오탐률이 0.1%가 되도록 임계값을 맞춘 것 (관리자가 로그를 보고 조정한 경우에 해당).

In [ ]:
def thr_at_fpr(scores_neg, target):
    s = np.sort(scores_neg)[::-1]
    k = int(np.floor(target * len(s)))
    return s[k] + 1e-9 if k < len(s) else s[-1]

def metrics(name, yt, pred, extra=None):
    yt, pred = np.asarray(yt).astype(int), np.asarray(pred).astype(bool)
    tp = int((pred & (yt == 1)).sum()); fn = int((~pred & (yt == 1)).sum())
    fp = int((pred & (yt == 0)).sum()); tn = int((~pred & (yt == 0)).sum())
    tpr = tp / max(tp + fn, 1); fpr = fp / max(fp + tn, 1); prec = tp / max(tp + fp, 1)
    f1 = 2 * prec * tpr / max(prec + tpr, 1e-9)
    d = dict(model=name, detection_rate=round(tpr, 4), false_positive_rate=round(fpr, 5),
             precision=round(prec, 4), f1=round(f1, 4), TP=tp, FN=fn, FP=fp, TN=tn)
    if extra:
        d.update(extra)
    return d

DEFAULT = {'rate': 10 * WIN, 'conc': 50}
neg_va = va & (W.label == 0).values
TUNED = {'rate': float(np.ceil(thr_at_fpr(W.srv_new_conn[neg_va].values, 0.001))),
         'conc': float(np.ceil(thr_at_fpr(W.srv_concurrent[neg_va].values, 0.001)))}
print('기본 임계값:', DEFAULT, ' / 튜닝 임계값(검증 오탐 0.1%):', TUNED)
for nm, th in (('기본', DEFAULT), ('튜닝', TUNED)):
    W[f'R1 새 연결 ({nm})'] = W.srv_new_conn >= th['rate']
    W[f'R2 동시 연결 ({nm})'] = W.srv_concurrent >= th['conc']
    W[f'R1 또는 R2 ({nm})'] = W[f'R1 새 연결 ({nm})'] | W[f'R2 동시 연결 ({nm})']
RULES = [c for c in W.columns if c.startswith('R1') or c.startswith('R2')]
yt = W.label[te].values
rows = [metrics(r, yt, W[r][te], {'kind': 'rule'}) for r in RULES]
print(pd.DataFrame(rows)[['model', 'detection_rate', 'false_positive_rate', 'TP', 'FN', 'FP', 'TN']].to_string(index=False))

# 정상인데 기본 룰에 걸린 IP — 관리자라면 화이트리스트를 검토할 대상
fp_ip = W[(W.label == 0) & W['R1 또는 R2 (기본)']].groupby('src_ip').agg(windows=('w', 'size'), max_new_conn=('srv_new_conn', 'max'),
                                                                     max_concurrent=('srv_concurrent', 'max')).sort_values('windows', ascending=False)
print('\n기본 룰에 걸린 정상 IP:'); print(fp_ip.head(10).to_string())
fp_ip.to_csv(os.path.join(RESULTS, 'rule_default_false_positive_ips.csv'))

## 9. AI 모델 A (그래디언트 부스팅) · B (MLP)
IP 주소와 시각은 특성에 넣지 않습니다.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import torch.nn as nn
FEATURES = ['new_conn', 'concurrent', 'srv_new_conn', 'srv_concurrent', 'dst_ips', 'dst_ports', 'no_reply', 'syn_only', 'rst', 'http', 'pkts_ts_mean', 'pkts_tc_mean',
            'bytes_ts_mean', 'bytes_tc_mean', 'bytes_ts_sum', 'dur_mean', 'dur_max']
X = W[FEATURES].astype('float64')
TRAIN_TIME = {}
def make_a():
    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, max_leaf_nodes=31, early_stopping=True,
                                          validation_fraction=0.1, random_state=SEED)
t0 = time.time(); A = make_a().fit(X[tr], W.label[tr]); TRAIN_TIME['A_gbdt'] = round(time.time() - t0, 1)
W['a_score'] = A.predict_proba(X)[:, 1]
print(f'A: 학습 {TRAIN_TIME["A_gbdt"]}초, 검증 AUC {roc_auc_score(W.label[va], W.a_score[va]):.4f}')

Xn = np.log1p(X.clip(lower=0)).values
mu, sd = Xn[tr].mean(0), Xn[tr].std(0) + 1e-6
Xn = ((Xn - mu) / sd).astype('float32')
model = nn.Sequential(nn.Linear(Xn.shape[1], 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1)).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
lossf = nn.BCEWithLogitsLoss()
Xt, ytt = torch.tensor(Xn[tr]), torch.tensor(W.label[tr].values, dtype=torch.float32)
def predict(Xa):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(torch.tensor(Xa).to(DEVICE)).squeeze(-1)).cpu().numpy()
t0 = time.time(); best_auc, best = 0, None
for ep in range(int(os.environ.get('DDOS_EPOCHS', 30))):
    model.train(); perm = torch.randperm(len(Xt))
    for i in range(0, len(perm), 512):
        b = perm[i:i+512]
        opt.zero_grad(); loss = lossf(model(Xt[b].to(DEVICE)).squeeze(-1), ytt[b].to(DEVICE)); loss.backward(); opt.step()
    auc = roc_auc_score(W.label[va], predict(Xn[va]))
    if auc > best_auc:
        best_auc, best = auc, {k: v.detach().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best); TRAIN_TIME['B_mlp'] = round(time.time() - t0, 1)
W['b_score'] = predict(Xn)
print(f'B: 학습 {TRAIN_TIME["B_mlp"]}초 ({DEVICE}), 최고 검증 AUC {best_auc:.4f}')

from sklearn.inspection import permutation_importance
pi = permutation_importance(A, X[te], W.label[te], scoring='roc_auc', n_repeats=5, random_state=SEED)
IMPORTANCE = pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False)
print('\n모델 A 순열 중요도:'); print(IMPORTANCE.head(8).round(4).to_string())

# 지름길 확인: 이 데이터에서 피해 웹서버로 가는 정상 HTTP가 거의 없어 'HTTP 비율'만으로 공격이 갈릴 수 있음 → 빼고 다시 학습
FEAT_NOHTTP = [f for f in FEATURES if f != 'http']
A2 = make_a().fit(X.loc[tr, FEAT_NOHTTP], W.label[tr])
W['a2_score'] = A2.predict_proba(X[FEAT_NOHTTP])[:, 1]
print(f"A' (HTTP 비율 제외): 검증 AUC {roc_auc_score(W.label[va], W.a2_score[va]):.4f}")
print(W.groupby('attack').http.mean().round(3).to_string())
IMPORTANCE.to_csv(os.path.join(RESULTS, 'feature_importance_A.csv'))

## 10. 비교 — 같은 오탐률에서 탐지율

In [ ]:
THRESH = {}
for col, nm in (('a_score', 'A GBDT'), ('a2_score', "A' GBDT(HTTP 제외)"), ('b_score', 'B MLP')):
    for tname, target in (('FPR 1%', 0.01), ('FPR 0.1%', 0.001)):
        th = thr_at_fpr(W[col][neg_va].values, target)
        THRESH[f'{nm} @ {tname}'] = float(th)
        rows.append(metrics(f'{nm} @ {tname}', yt, W[col][te] >= th, {'kind': 'ai'}))
thA = THRESH['A GBDT @ FPR 0.1%']
W['hybrid'] = W['R1 또는 R2 (튜닝)'] | (W.a_score >= thA)
rows.append(metrics('Hybrid: R1 또는 R2(튜닝) OR A@0.1%', yt, W.hybrid[te], {'kind': 'hybrid'}))
cmp = pd.DataFrame(rows)
pd.set_option('display.width', 220)
print(cmp[['model', 'detection_rate', 'false_positive_rate', 'precision', 'TP', 'FN', 'FP', 'TN']].to_string(index=False))
cmp.to_csv(os.path.join(RESULTS, 'comparison.csv'), index=False)

# 웹서버 관점: 피해 서버(192.168.10.50)로 연결한 구간만 — 서버 관리자가 실제로 보게 되는 트래픽
sv = te & W.to_victim.values.astype(bool)
ysv = W.label[sv].values
srv_rows = [metrics(r, ysv, W[r][sv]) for r in ('R1 또는 R2 (기본)', 'R1 또는 R2 (튜닝)')]
for col, nm in (('a_score', 'A GBDT'), ('a2_score', "A' GBDT(HTTP 제외)")):
    srv_rows.append(metrics(f'{nm} @ FPR 0.1%', ysv, W[col][sv] >= THRESH[f'{nm} @ FPR 0.1%']))
SRV = pd.DataFrame(srv_rows)
print(f'\n웹서버로 간 시험 구간만 (정상 {int((ysv==0).sum())} / 공격 {int(ysv.sum())}):')
print(SRV[['model', 'detection_rate', 'false_positive_rate', 'TP', 'FN', 'FP', 'TN']].to_string(index=False))
SRV.to_csv(os.path.join(RESULTS, 'comparison_webserver_view.csv'), index=False)

## 11. 공격 종류별 — 홍수형 vs 느린 공격
Hulk·GoldenEye·LOIT는 요청을 쏟아붓는 **홍수형**, slowloris·Slowhttptest는 연결을 적게 열고 오래 붙잡는 **느린(저속) 공격**입니다.

In [ ]:
tt = W[te]
per = tt.groupby('attack').agg(windows=('label', 'size'), srv_new_conn_med=('srv_new_conn', 'median'), srv_concurrent_med=('srv_concurrent', 'median'),
                               dur_mean_med=('dur_mean', 'median'),
                               R1_default=('R1 새 연결 (기본)', 'mean'), R2_default=('R2 동시 연결 (기본)', 'mean'),
                               rule_default=('R1 또는 R2 (기본)', 'mean'), rule_tuned=('R1 또는 R2 (튜닝)', 'mean'),
                               ai_A=('a_score', lambda s: float((s >= thA).mean())),
                               ai_A_nohttp=('a2_score', lambda s: float((s >= THRESH["A' GBDT(HTTP 제외) @ FPR 0.1%"]).mean())))
print(per.round(3).to_string())
per.to_csv(os.path.join(RESULTS, 'by_attack.csv'))

## 12. 처음 보는 공격 — 공격 종류 하나씩 빼고 학습
임계값 룰은 공격 종류를 몰라도 "많이 붙으면" 막습니다. AI는 어떤가요? 공격 하나를 학습에서 통째로 빼고 다시 학습해 봅니다.

In [ ]:
loo = []
for atk in sorted(set(tt.attack) - {'BENIGN'}):
    keep = tr & (W.attack != atk).values
    m = make_a().fit(X[keep], W.label[keep])
    th = thr_at_fpr(m.predict_proba(X[neg_va])[:, 1], 0.001)
    s = m.predict_proba(X[te & (W.attack == atk).values])[:, 1]
    m2 = make_a().fit(X.loc[keep, FEAT_NOHTTP], W.label[keep])
    th2 = thr_at_fpr(m2.predict_proba(X.loc[neg_va, FEAT_NOHTTP])[:, 1], 0.001)
    s2 = m2.predict_proba(X.loc[te & (W.attack == atk).values, FEAT_NOHTTP])[:, 1]
    loo.append({'attack': atk, 'test_windows': int((tt.attack == atk).sum()), 'A_seen': round(float(per.loc[atk, 'ai_A']), 4),
                'A_unseen': round(float((s >= th).mean()), 4), 'A_nohttp_unseen': round(float((s2 >= th2).mean()), 4), 'rule_default': round(float(per.loc[atk, 'rule_default']), 4),
                'rule_tuned': round(float(per.loc[atk, 'rule_tuned']), 4)})
    print(loo[-1])
LOO = pd.DataFrame(loo); LOO.to_csv(os.path.join(RESULTS, 'unseen_attack.csv'), index=False)

## 13. 공격이 시작되고 몇 초 만에 잡았나
공격 종류별로 **첫 공격 구간**부터 각 방식이 처음 탐지한 구간까지의 시간을 봅니다 (시험 집합이 아닌 전체 구간 기준, AI는 오탐 0.1% 기준값).

In [ ]:
W['ai_A_flag'] = W.a_score >= thA
ttd = []
for atk, g2 in W[W.label == 1].groupby('attack'):
    first = g2.w.min()
    row = {'attack': atk, 'attack_windows': len(g2), 'span_min': round((g2.w.max() - first + 1) * WIN / 60, 1)}
    for col in ('R1 또는 R2 (기본)', 'R1 또는 R2 (튜닝)', 'ai_A_flag'):
        hit = g2[g2[col]].w
        row[col] = None if hit.empty else int((hit.min() - first) * WIN)
    ttd.append(row)
TTD = pd.DataFrame(ttd); print(TTD.to_string(index=False))
TTD.to_csv(os.path.join(RESULTS, 'time_to_detect.csv'), index=False)

## 14. 그래프와 저장

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
fig, ax = plt.subplots(figsize=(7.5, 5.5))
for col, nm in (('a_score', 'A GBDT'), ('a2_score', "A' GBDT (no HTTP share)"), ('b_score', 'B MLP')):
    fpr, tpr, _ = roc_curve(yt, W[col][te]); ax.plot(fpr, tpr, label=f'{nm} (AUC {roc_auc_score(yt, W[col][te]):.3f})')
for col, nm, ls in (('srv_new_conn', 'R1 new conns to one server (sweep)', '--'), ('srv_concurrent', 'R2 concurrent to one server (sweep)', ':')):
    fpr, tpr, _ = roc_curve(yt, W[col][te]); ax.plot(fpr, tpr, ls, c='gray', label=nm)
for r, mk in (('R1 또는 R2 (기본)', 'o'), ('R1 또는 R2 (튜닝)', 's')):
    m = cmp[cmp.model == r].iloc[0]
    ax.scatter(m.false_positive_rate, m.detection_rate, marker=mk, s=80, zorder=5, label='R1 or R2 ' + ('default' if '기본' in r else 'tuned'))
ax.set_xscale('symlog', linthresh=0.01); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.set_xlabel('False positive rate (log scale above 1%)'); ax.set_ylabel(f'Detection rate (per source IP x {WIN}s window)')
ax.set_title('CIC-IDS2017 DoS/DDoS: threshold rules vs AI'); ax.grid(alpha=0.3); ax.legend(fontsize=8, loc='lower right')
plt.tight_layout(); plt.savefig(os.path.join(RESULTS, 'roc.png'), dpi=130); plt.show()

summary = {
    'dataset': f'CIC-IDS2017 {"+".join(DAYS)}', 'window_sec': WIN, 'pcap_gb': PCAP_GB, 'suricata_sec': SURI_SEC,
    'flow_labels': FL.label.value_counts().to_dict(), 'window_stats': WINDOW_STATS,
    'split': W.groupby(['split', 'attack']).size().unstack(fill_value=0).to_dict(),
    'schedule_check': SCHED_CHECK.to_dict(orient='records'), 'versions': VERSIONS, 'thresholds_default': DEFAULT, 'thresholds_tuned': TUNED, 'ai_thresholds': THRESH,
    'train_time_sec': TRAIN_TIME, 'importance_A': IMPORTANCE.round(4).to_dict(),
    'comparison': cmp.to_dict(orient='records'), 'webserver_view': SRV.to_dict(orient='records'), 'by_attack': per.reset_index().to_dict(orient='records'),
    'unseen_attack': LOO.to_dict(orient='records'), 'time_to_detect': TTD.to_dict(orient='records'),
}
json.dump(summary, open(os.path.join(RESULTS, 'summary.json'), 'w'), ensure_ascii=False, indent=2, default=str)
W.drop(columns=['src_ip']).to_csv(os.path.join(RESULTS, 'windows.csv.gz'), index=False)
zip_path = shutil.make_archive(os.path.join(WORK, 'ddos_lab_results'), 'zip', RESULTS)
print('저장:', zip_path, f'{os.path.getsize(zip_path)/1e6:.1f} MB'); print(sorted(os.listdir(RESULTS)))
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)